In [4]:
# Libraries required to install before executing this model

# pip install numpy
# pip install pandas
# pip install torch
# pip install PyPDF
# pip install faiss-gpu
# pip install tranformers
# pip install sentence-transformers
# pip install sentencepiece

In [14]:
import numpy as np

In [15]:
import faiss

In [16]:
from sentence_transformers import SentenceTransformer

In [17]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2301.95it/s]


In [18]:
from transformers import pipeline

In [19]:
import re

In [39]:
import json

In [20]:
from huggingface_hub import login
login()

In [145]:
from pypdf import PdfReader

def read_pdf_pypdf(file_path):
    try:
        reader = PdfReader(file_path)
        text = ""
        for page in reader.pages:
            text += page.extract_text() or ""  # Handle pages with no text
        return text.strip()
    except FileNotFoundError:
        return "Error: File not found."
    except Exception as e:
        return f"Error reading PDF: {e}"

# Example usage
if __name__ == "__main__":
    pdf_text = read_pdf_pypdf("SushrutaS.pdf")
    #print(pdf_text)

In [146]:
def naive_code_chunk(text, chunk_size=300):
    """Slices plain text without considering code structure."""
    return [text[i:i + chunk_size] for i in range(0, len(text), chunk_size)]

simple_chunks = naive_code_chunk(pdf_text, chunk_size=300)

In [147]:
#type(simple_chunks)

In [148]:
with open("SushrutaS.json", "w") as file:
    json.dump(simple_chunks, file, indent=4)

In [149]:
#print(simple_chunks[0])

In [150]:
embeddings = model.encode(
    simple_chunks,
    convert_to_numpy=True,
    normalize_embeddings=True,  # Useful for cosine similarity
    show_progress_bar=True
)

Batches: 100%|██████████| 82/82 [01:06<00:00,  1.23it/s]


In [175]:
# embeddings = model.encode(
#     objective,
#     convert_to_numpy=True,
#     normalize_embeddings=True,  # Useful for cosine similarity
#     show_progress_bar=True
# )

In [176]:
print("Embedding shape:", embeddings.shape)
print("First embedding length:", len(embeddings[0]))

Embedding shape: (5, 384)
First embedding length: 384


In [177]:
embeddings = embeddings.astype("float32")

In [178]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)

In [179]:
index.add(embeddings)

In [180]:
faiss.write_index(index, "Objective_index.faiss")

In [156]:
#index = faiss.read_index("Aryu1_index.faiss")

In [157]:
print("Number of vectors in index:", index.ntotal)

Number of vectors in index: 2604


In [159]:
query = input("Enter your query: ")

# Embed the query
query_embedding = model.encode(
    [query],
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

# Search top 3 similar chunks
k = 5
scores, indices = index.search(query_embedding, k)
# print("Top Results:")
# for score, idx in zip(scores[0], indices[0]):
#     print(f"Score: {score:.4f} and indices: {indices[0]}")
#     print("Text:", simple_chunks[idx])
#     print()

Enter your query:  Which Ayurvedic concepts are explained in detail?


In [160]:
scores

array([[0.5450109 , 0.5394841 , 0.52610064, 0.52192104, 0.51256776]],
      dtype=float32)

In [161]:
indices[0][0]

np.int64(455)

In [162]:
simple_chunks[indices[0][0]]

'branchesofstudytomen.Inthepresentscience(Ayurveda),thePiirusha(self-consciousoro:anicindividual)isdescribedastheresultantofthecombinationofthesoulandthefiveprimar>-materialprinciples.Allmedicalactssuchas,surgicaloperations,administrationofmedicinalremediesandapplicationsofalkalinesubstances,orcauter'

In [163]:
#objective=[]
objective.append(simple_chunks[indices[0][0]])

In [173]:
# with open("objective.json", "w") as file:
#     json.dump(objective, file, indent=4)

In [165]:
objective

['ture of therapeutics in India is the widespread\npopularity of ‘alternative’ systems of medicine among patients.\nAyurveda, Siddha, Unani, Yoga, Naturopathy and Homeopathy\nare officially recognised by the Government of India and most\nclinicians will come across patients, especially those with\nchronic ',
 'CENTRAL COUNCIL FOR RESEARCH IN AYURVEDA AND SIDDHA\nDepartment of ISM&H, Ministry of Health &\nFamily Welfare, Govt of India\ni\nAYURVEDIC HOME REMEDIES\nCENTRAL COUNCIL FOR RESEARCH  \nIN AYURVEDA AND SIDDHA\nDepartment of ISM& H, Ministry of Health \n& Family Welfare, Govt of India\n©  Central C ou ncil f',
 's, to take a scientiﬁc approach to the presentation of both the Ayurvedic and\nconventional medicine data on a particular topic .  \nFinally, I thank my children, Aparna Rani, Gyan Sagar, Anita Rani, and Dhyan Sagar,\nfor their encouragement during the preparation of this book. \n \nLakshmi Chandra Mishr',
 ' requests Ayurvedic scholars, doctors and students to \ncommunic

In [166]:
retrieved_chunks = [simple_chunks[i] for i in indices[0]]

In [167]:
print("Retrieved Context:\n")
for i, chunk in enumerate(retrieved_chunks, 1):
    print(f"{i}. {chunk}\n")

Retrieved Context:

1. branchesofstudytomen.Inthepresentscience(Ayurveda),thePiirusha(self-consciousoro:anicindividual)isdescribedastheresultantofthecombinationofthesoulandthefiveprimar>-materialprinciples.Allmedicalactssuchas,surgicaloperations,administrationofmedicinalremediesandapplicationsofalkalinesubstances,orcauter

2. sa-Sthanam(Therapeutics)andKalpa-Sthanam(Toxicology.Subjectsotherthantheprecedingoneswillbediscussedintheclosingchaptersofthebookbywa}-ofanAppendix(Uttara-Tantranv.Authoritativeverseonthesub-ject:—ThemanwhoreadsthisEternalScienceofMedicine(Ayurveda-Shastram)discoursedbytheself-originedBrahmaandpropa

3. al.SuchthingsastheMrinalam(thestemofalotusplant),theVisham(thelotusbulb),Shaluka,Kandaandsugar-caneshouldbeeatenatthebeginningofameal,andneveratitsclose.Aman,whoiswell-readintheScienceofmedicine(Ayurveda),shouldsitduringhismealinaneasypostureonahighseatandpartake,attherighttime,withhisbodyerectandhi

4. am.Chap.V.
xn •INTRODUCTION.wisdomancfexperienge.D'allanahasma

In [168]:
context = "\n\n".join(retrieved_chunks)

In [169]:
augmented_prompt = f"""
You are a helpful assistant.

Use the following context to answer the user's question.
If the answer is not contained in the context, say you don't know.

Context:
{context}

Question:
{query}

Answer:
"""

In [170]:
augmented_prompt

"\nYou are a helpful assistant.\n\nUse the following context to answer the user's question.\nIf the answer is not contained in the context, say you don't know.\n\nContext:\nbranchesofstudytomen.Inthepresentscience(Ayurveda),thePiirusha(self-consciousoro:anicindividual)isdescribedastheresultantofthecombinationofthesoulandthefiveprimar>-materialprinciples.Allmedicalactssuchas,surgicaloperations,administrationofmedicinalremediesandapplicationsofalkalinesubstances,orcauter\n\nsa-Sthanam(Therapeutics)andKalpa-Sthanam(Toxicology.Subjectsotherthantheprecedingoneswillbediscussedintheclosingchaptersofthebookbywa}-ofanAppendix(Uttara-Tantranv.Authoritativeverseonthesub-ject:—ThemanwhoreadsthisEternalScienceofMedicine(Ayurveda-Shastram)discoursedbytheself-originedBrahmaandpropa\n\nal.SuchthingsastheMrinalam(thestemofalotusplant),theVisham(thelotusbulb),Shaluka,Kandaandsugar-caneshouldbeeatenatthebeginningofameal,andneveratitsclose.Aman,whoiswell-readintheScienceofmedicine(Ayurveda),shouldsitdurin

In [171]:
generator = pipeline(
    "text-generation",
    model="google/gemma-2-2b-it"
)

response = generator(
    augmented_prompt,
    max_new_tokens=800,
    do_sample=False,
    eos_token_id=generator.tokenizer.eos_token_id,
)
text = response[0]["generated_text"]

sentences = re.findall(r'.*?[.!?](?=\s|$)', text, flags=re.DOTALL)
text = "".join(sentences)

print(text)
#print(response[0]["generated_text"])

Loading weights: 100%|██████████| 288/288 [00:00<00:00, 3138.91it/s]
[transformers] Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



You are a helpful assistant.

Use the following context to answer the user's question.
If the answer is not contained in the context, say you don't know.

Context:
branchesofstudytomen.Inthepresentscience(Ayurveda),thePiirusha(self-consciousoro:anicindividual)isdescribedastheresultantofthecombinationofthesoulandthefiveprimar>-materialprinciples.Allmedicalactssuchas,surgicaloperations,administrationofmedicinalremediesandapplicationsofalkalinesubstances,orcauter

sa-Sthanam(Therapeutics)andKalpa-Sthanam(Toxicology.Subjectsotherthantheprecedingoneswillbediscussedintheclosingchaptersofthebookbywa}-ofanAppendix(Uttara-Tantranv.Authoritativeverseonthesub-ject:—ThemanwhoreadsthisEternalScienceofMedicine(Ayurveda-Shastram)discoursedbytheself-originedBrahmaandpropa

al.SuchthingsastheMrinalam(thestemofalotusplant),theVisham(thelotusbulb),Shaluka,Kandaandsugar-caneshouldbeeatenatthebeginningofameal,andneveratitsclose.Aman,whoiswell-readintheScienceofmedicine(Ayurveda),shouldsitduringhismealinan